[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/row-processing-system/blob/main/notebooks/controller_colab.ipynb)

# Row processing controller

This notebook runs the controller on a CPU runtime. The controller sends "write a row" jobs to model APIs, checks the rows, hands them to review agents, reads their verdicts and ships accepted rows to a private dataset repository.

Every step is written to an event log. You can stop at any moment (Runtime > Interrupt execution, or create the STOP file named in the run cell) and run the cells again: the run resumes where it left off, and no call or row is repeated.

Run the cells in order:

1. Install the package. After this cell Colab may ask to restart the session (the install replaces packages Colab had already imported at start). Restart it, then go on from cell 2.
2. Map your Colab secrets to environment variables (only the secret names are printed).
3. Optionally try one real route with a handful of calls (the dry-run cell).
4. Set the configuration.
5. Write the tasks file into the state store (an example the first time).
6. Run the controller for a time budget.
7. Print the status.

Colab secrets used (add them in the Secrets panel and allow this notebook to read them): `CLAUDE_API`, `OPENAI_API`, `GEMINI_API`, `DEEPSEEK_API`, `HF_TOKEN`.

The configuration below is a dry run: a pretend provider and the runtime's own disk, so it needs no keys. For real work, name a private storage bucket and a private dataset repository and set routes for real providers (version 0.2.8 has adapters for Anthropic, OpenAI, Gemini and DeepSeek; try a route with the dry-run cell first).

Two things to know for real work. GPT is used one call at a time in this project (a route of kind `api`, such as `luna-sync`), never through the batch API. And test runs and production should not share state: give each its own folder inside the bucket, such as `owner/rps-state/test` and `owner/rps-state/prod`.

In [ ]:
# Install the package from GitHub at a fixed release tag.
# After this cell Colab may ask to restart the session (the install replaces packages Colab
# imported at start). Restart, then go on from cell 2; this cell does not need to run again.
!pip install "git+https://github.com/AbstractEyes/row-processing-system@v0.2.8"

In [ ]:
# Copy each Colab secret into its environment variable:
#   CLAUDE_API -> ANTHROPIC_API_KEY      OPENAI_API -> OPENAI_API_KEY
#   GEMINI_API -> GEMINI_API_KEY         DEEPSEEK_API -> DEEPSEEK_API_KEY
#   HF_TOKEN   -> HF_TOKEN
# Only a secret's NAME is printed, never its value. A secret that is missing, or that
# this notebook is not allowed to read, is reported as "not available".
from rps.keys import map_colab_secrets

ready = map_colab_secrets()

In [ ]:
# Optional: try ONE real route before you use it. This sends real calls (a few cents at most),
# waits for the answers, and prints them with the token counts and the cost. It
# refuses to start if the estimate would pass the test cap. Routes: luna-sync (GPT is used one
# call at a time in this project), haiku-batch, flash-batch, deepseek-sync, and the one-call
# twins haiku-sync and flash-sync. The key for the route must be in the secrets mapped above.
# Leave the name as None to skip.
from rps.dryrun import dry_run

DRY_RUN_ROUTE = None  # for example "luna-sync"
if DRY_RUN_ROUTE:
    report = dry_run(DRY_RUN_ROUTE, calls=5)

In [ ]:
# The configuration holds no keys. Edit the values below, or set CONFIG_PATH to a JSON file.
import rps
from rps.config import Config

CONFIG_PATH = ""  # for example "/content/drive/MyDrive/rps/config.json"
WORK_DIR = "/content/rps_work"
TIME_BUDGET_SECONDS = 20 * 60  # how long one run may work before it stops (it can be resumed)

CONFIG = {
    # Where the state lives. With both left as None this is a dry run on the runtime's own
    # disk, which is lost when the runtime is recycled. For real work name a private storage
    # bucket (event logs, review packets and pictures, the stop file and the tasks file) and a
    # private dataset repository (accepted rows and the ledger), such as "owner/rps-state" and
    # "owner/rows". Both are created, private, if they do not exist yet.
    # The bucket may name a folder inside it, and every state file then lives in that folder.
    # Use one folder for test runs and another for production so they never share a file:
    # "owner/rps-state/test" and "owner/rps-state/prod". The reviewers' config file must name
    # the same bucket and folder as the run.
    "bucket": None,
    "data_repo": None,
    "work_dir": WORK_DIR,
    # The tasks file is a file in the state store (under the bucket's folder when there is one).
    # The controller reads it again on every pass, and a missing file means no tasks.
    "tasks_file": "store:control/tasks.json",
    # One route per way of getting rows written. Each route has a total budget in dollars and
    # a cap for every purpose a call can have (test, pricing, probe, rebound, production).
    "routes": {
        "demo": {
            "provider": "fake",
            "model": "fake-model",
            "kind": "batch",
            "budget_usd": 1.0,
            "purpose_caps": {"test": 0.1, "pricing": 0.1, "probe": 0.1, "rebound": 0.2, "production": 0.5},
        },
        # A real route looks like this (its key must be in the secrets). kind is "batch" or "api"
        # (one call at a time); "effort" is the provider's reasoning or thinking level.
        # GPT is used one call at a time in this project, so its route is kind "api", not "batch".
        # "luna-sync": {
        #     "provider": "openai", "model": "gpt-6-luna", "kind": "api", "effort": "medium",
        #     "budget_usd": 5.0, "purpose_caps": {"test": 0.5, "rebound": 1.0, "production": 4.0},
        #     # "workers": 32,  # calls in flight at once; 32 is the default for a one-call route
        # },
    },
    # Pictures go only to providers the routing table allows for their class: say which class a
    # source's pictures are (public_common, coco, ffhq, deepfashion, hico) with "image_class".
    "sources": {"demo-source": {"license": "CC0-1.0"}},
    # The writing rules are an input. A real run uses the shipped rule pool and its assessor;
    # the pretend provider above does not need them.
    # "rules": {"package": "rps.rules"},
    # Reviewers read the packets the controller publishes. A row's first review goes to a
    # reviewer from a model family other than its writer's.
    "review": {
        "packet_size": 10,
        "reviewers": [
            {"id": "reviewer-1", "family": "anthropic"},
            {"id": "reviewer-2", "family": "openai"},
        ],
    },
}

config = Config.load(CONFIG_PATH) if CONFIG_PATH else Config.from_dict(CONFIG)
print("rps version", rps.__version__)
print("routes:", ", ".join(sorted(config.routes)))
for note in config.problems():
    print("note:", note)

In [ ]:
# A task is a batch of spec lines for one source, written through one route for one purpose.
# The tasks file lives in the state store, so a run reads it again on every pass and you can add
# tasks while it goes on, from another session or script: write the file as the operator, as this
# cell does. This writes a small example the first time; replace it with your own. The controller
# skips tasks it has already recorded, so the file can grow.
import json

from rps.runtime import build_operator_store

tasks_path = config.tasks_store_path()  # "control/tasks.json" for the setting above
if tasks_path is None:
    print("tasks file:", config.tasks_file, "(a local file, not in the state store: write your tasks there)")
else:
    operator = build_operator_store(config, create=True)
    if not operator.exists(tasks_path):
        example = [
            {
                "source": "demo-source",
                "batch": "demo-1",
                "route": "demo",
                "purpose": "test",
                "lines": [{"line_id": f"{n:03d}", "spec": f"Write one short thought about item {n}."} for n in range(1, 7)],
            }
        ]
        operator.write_text(tasks_path, json.dumps(example, indent=1))
        operator.flush(force=True)  # a bucket holds writes back; this makes the file durable now
    print("tasks file:", config.tasks_file)

In [ ]:
# Build the controller and run it until nothing is left to do, the time budget ends, a STOP
# file appears, or you interrupt the cell. Interrupting is safe: run this cell again to resume.
# To stop from outside, create the file named below (or control/STOP in the state store, which is
# under the bucket's folder when there is one). A dataset repository takes one commit a minute, so
# a run ends up to a minute after its last change; the progress line shows the files waiting for it.
from rps.runtime import build_controller

controller = build_controller(config, create=True)
print("stop file:", config.stop_file_path())
summary = controller.run(time_budget_seconds=TIME_BUDGET_SECONDS)
print("run ended:", summary.reason)

In [ ]:
# Where things stand: progress, state counts, and spend per route and purpose against its cap.
print(controller.format_status())

## After a run

Rows wait for their reviewers once a run ends. A reviewer pulls its packet, records verdicts and pushes them (this burns the packet's pictures on the reviewer's machine):

    python -m rps.review pull --reviewer reviewer-1 --config config.json --into packets
    python -m rps.review verdict --packet PACKET --row ROW --verdict keep
    python -m rps.review push --packet PACKET --config config.json

Reviewers reach the packets through the storage bucket, so name a bucket in the configuration before using real reviewers. The config file you give the reviewer commands must name the same bucket, folder included, as the run. Run the run cell again afterwards: the controller reads the verdicts, settles each row, sends questioned rows to a further reviewer, and ships the accepted rows.